# Kaggle Netflow Graph Dataset: Construction and Visualization

This notebook covers **only the pre-built Kaggle graph dataset** (`data/kaggle/`).
The companion notebook, `02_graph_construction_and_visualization_ids_2017.ipynb`,
covers the CIC-IDS-2017 raw flow logs separately. Splitting them keeps each
notebook focused on one data source, so a reader is never unsure which
dataset a given plot or number came from.

**What this dataset is.** Three GraphML files, all covering the same
41,073 IP addresses, built from 100,000 sampled NetFlow-style records
(the "0.1M" in the filenames). Each edge carries 7 z-scored (standardized)
numeric features - packet count, byte counts, duration, protocol, direction,
connection state - plus a ground-truth `ActivityLabel` (0 = benign,
1 = malicious).

**What this notebook answers, precisely enough to present and defend to the
class:**
1. What exactly is different between the "plain", "multi", and "aggregated"
   files - with real numbers, not assumptions.
2. What does this graph's structure actually look like, and why (degree
   distribution, hub dominance, connected components)?
3. Where do the attacks sit structurally - on the busiest nodes, or
   elsewhere?
4. How do the standard centrality features behave on a graph shaped like
   this one?

Every claim below is backed by a computed number in the cell above it -
if you re-run this notebook, you should get the same numbers.

In [1]:
import sys
!{sys.executable} -m pip install -r ../requirements.txt

%matplotlib inline

sys.path.append("../src")
print("Using interpreter:", sys.executable)

import networkx as nx
import pandas as pd
from collections import Counter
from load_kaggle_graph import load_graphml, mark_malicious_nodes, summarize_graph, compare_graph_variants
from graph_features import compute_all_features, verify_features
from visualize import (
    plot_graph, plot_multiple_graphs, plot_degree_distribution, plot_feature_distribution,
    get_readable_subgraph, get_subgraph_excluding_nodes, plot_bar, plot_scatter_2d,
    plot_model_comparison, BENIGN_COLOR, MALICIOUS_COLOR,
)

OverflowError: cannot convert longdouble infinity to integer

In [ ]:
pd.set_option("display.max_columns", 12)

KAGGLE_DIR = "../data/kaggle"
VARIANTS = {
    "plain":      f"{KAGGLE_DIR}/0.1M-Stratified.graphml",
    "multi":      f"{KAGGLE_DIR}/0.1M-Stratified-Multi.graphml",
    "aggregated": f"{KAGGLE_DIR}/0.1M-Stratified-aggregated.graphml",
}

## 1. Loading all three variants and comparing them directly

Each file is on the order of 20-40MB (~41K nodes) so all three fit in
memory comfortably - no sampling needed for this step.

In [ ]:
comparison = compare_graph_variants(VARIANTS)
comparison

**Reading this table:** "plain" and "aggregated" have identical node and
edge counts (45,221 edges - one per unique IP pair). "multi" has more than
double that (100,000 edges - the ActivityLabel/multigraph. It's the only one that keeps
every individual flow as its own parallel edge, matching the "0.1M" =
100,000 original flow records exactly.

In [ ]:
graphs = {name: mark_malicious_nodes(load_graphml(path)) for name, path in VARIANTS.items()}
G, G_multi, G_agg = graphs["plain"], graphs["multi"], graphs["aggregated"]

for name, G_ in graphs.items():
    print(name, "->", summarize_graph(G_, name=name))

## 2. What "plain" vs "aggregated" actually means, with real numbers

Both files store exactly one edge per unique (source, destination) pair.
The question is: when two IPs exchanged more than one flow, what value
ends up on that single edge?

We pick a real pair from this dataset that had **16** separate flows
between them in the "multi" file, and compare all three representations
directly.

In [ ]:
hub = max(G.degree, key=lambda x: x[1])[0]
print("Busiest node in this graph:", hub, "- degree:", G.degree(hub))

# find a neighbor of the hub that had several parallel flows in the multi-graph
parallel_counts = Counter()
for u, v, k in G_multi.edges(hub, keys=True):
    other = v if u == hub else u
    parallel_counts[other] += 1
busy_neighbor, n_parallel = parallel_counts.most_common(1)[0]
print(f"Neighbor with the most repeated flows to the hub: {busy_neighbor} ({n_parallel} flows)")

plain_edge = dict(G[hub][busy_neighbor])
agg_edge = dict(G_agg[hub][busy_neighbor])
multi_values = [float(d.get("TotPkts")) for d in G_multi[hub][busy_neighbor].values()]

print("\nPLAIN edge TotPkts      :", plain_edge.get("TotPkts"))
print("AGGREGATED edge TotPkts :", agg_edge.get("TotPkts"))
print(f"\nAcross the {n_parallel} individual MULTI flows for this pair, TotPkts values were:")
print([round(v, 6) for v in multi_values])
print("Mean of those values   :", sum(multi_values) / len(multi_values))

**Precise conclusion - don't overstate this:** "plain" and "aggregated"
give *different* values for pairs with more than one underlying flow (confirmed
above), so they are genuinely different representations, not duplicates.
However, the "aggregated" value is **not** simply the mean, median, or sum
of the individual flow values (check the numbers above - none of those
match exactly). The dataset does not document its exact aggregation
formula, so the honest statement for a presentation is: *"aggregated"
combines repeated flows between a pair into one edge using an unspecified
aggregation rule that is not a simple average* - not "aggregated = summed"
as one might assume from the name alone. When in doubt, say what you can
verify, not what seems intuitive.

## 3. Degree distribution: this graph is hub-dominated

Before drawing any node-link picture, look at how degree is distributed
across all 41,073 nodes. This is the single most important structural
fact about this graph, and it explains everything the plots below will
show.

In [ ]:
degrees = [d for _, d in G.degree()]
n_total = len(degrees)
hub_degree = G.degree(hub)

print(f"Total nodes: {n_total}")
print(f"Busiest node ({hub}) degree: {hub_degree}  ->  {100*hub_degree/sum(degrees):.1f}% of all degree in the graph")
print(f"Nodes with degree == 1 (a single one-off conversation): {sum(1 for d in degrees if d==1)} ({100*sum(1 for d in degrees if d==1)/n_total:.1f}%)")
print(f"Nodes with degree >= 100: {sum(1 for d in degrees if d>=100)}")
print(f"Nodes with degree >= 10:  {sum(1 for d in degrees if d>=10)}")

top5 = sorted(G.degree, key=lambda x: x[1], reverse=True)[:5]
print("\nTop 5 busiest nodes:")
for node, d in top5:
    print(f"  {node}: degree={d}, touched_an_attack_flow={G.nodes[node]['is_malicious']}")

In [ ]:
plot_degree_distribution(G, title="Kaggle Graph - Degree Distribution (all 41,073 nodes, log-log)",
                          save_path="figures/fig_03_degree_distribution.png")

**What this shows:** almost all of the graph (95.8% of nodes) has degree
1 - a single one-off conversation with someone else. Degree then falls off
sharply, with only 27 nodes reaching degree 100 or higher, and one single
node - `147.32.84.229` - accounting for **30% of all edges in the entire
graph**. This is a classic single-vantage-point NetFlow signature: the
data was almost certainly captured from (or centered on) one monitored
host or gateway, so that host appears as a conversation partner in a huge
fraction of all recorded flows. It is not a data quality problem, and it
is not something graph sampling introduced - it is a structural property
of the raw data itself, confirmed here using the complete, un-sampled
41,073-node graph.

**Why this matters for the project:** with a structure this skewed, raw
degree centrality will rank the gateway node #1 by an enormous margin and
say almost nothing else useful - nearly every node will otherwise look
identical (degree 1). Betweenness, PageRank, and clustering coefficient
(computed later in this notebook) are more informative here precisely
*because* they respond differently to hub-and-spoke structure than raw
degree does.

## 4. What the hub neighborhood looks like

`plot_graph` automatically keeps only the 40 highest-degree nodes when a
graph is this large, and labels the plot to say so - never silently
showing a subset as if it were the whole thing.

In [ ]:
plot_graph(G, title=f"Hub neighborhood - busiest node {hub}", max_nodes=40,
           save_path="figures/fig_04_hub_neighborhood.png")

## 5. Structure away from the main hub

`data/kaggle/` isn't *one* connected blob - it's a giant component plus many
small, separate fragments. Excluding the single dominant hub node reveals
the **second**-busiest node instead (still a strong hub, just an order of
magnitude smaller), which is worth seeing explicitly rather than assuming
the graph "flattens out" once the top node is removed.

In [ ]:
H = G.copy()
H.remove_node(hub)
second_hub, second_hub_degree = max(H.degree, key=lambda x: x[1])
print(f"With {hub} removed, the next busiest node is {second_hub} "
      f"- degree {second_hub_degree} in the remaining graph (unchanged from its original degree, since removing one other node barely affects it)")

G_no_hub = get_readable_subgraph(H, max_nodes=40)
plot_graph(G_no_hub, title=f"Same graph with the top hub ({hub}) removed", max_nodes=40,
           save_path="figures/fig_05_no_hub_structure.png")

## 6. Connected components: how many separate pieces does this graph have?

A graph can have a giant hub-dominated core AND many small, fully
disconnected fragments at the same time. Here's the exact breakdown.

In [ ]:
components = sorted(nx.connected_components(G), key=len, reverse=True)
sizes = [len(c) for c in components]

print(f"Total connected components: {len(components)}")
print(f"Giant component size: {sizes[0]} nodes ({100*sizes[0]/n_total:.1f}% of all nodes)")
print(f"Remaining {len(components)-1} components range from {min(sizes[1:])} to {max(sizes[1:])} nodes each")
print("Sizes of the 10 largest non-giant components:", sizes[1:11])

In [ ]:
small_component_nodes = sorted(components[1], key=str)
G_small = G.subgraph(components[1]).copy()
print(f"Showing one full non-giant component: {G_small.number_of_nodes()} nodes, {G_small.number_of_edges()} edges (no sampling needed - it's small enough to show completely)")

plot_graph(G_small, title="One isolated component, shown in full (not sampled)", max_nodes=G_small.number_of_nodes())

**Notice the shape repeats.** This small, fully separate 31-node
fragment is *also* a star: one node talking to 30 others who don't talk to
each other. The same one-host-to-many-peers pattern shows up at every
scale in this dataset - the giant hub, the second-tier hub, and even the
small disconnected fragments. That consistency is itself evidence this is
how the data was captured (from the perspective of individual hosts, each
one making many outbound connections), rather than a modeling artifact.

## 7. Where do the attacks sit structurally?

This is the question that actually matters for intrusion detection: are
malicious flows concentrated on the busiest nodes (easy to spot by degree
alone), or scattered among the quiet, one-off nodes (where degree tells
you nothing)?

In [ ]:
attack_edges = [(u, v) for u, v, d in G.edges(data=True) if float(d.get("ActivityLabel", 0)) != 0]
attack_nodes = set(n for edge in attack_edges for n in edge)

print(f"Attack-labeled edges: {len(attack_edges)} / {G.number_of_edges()} ({100*len(attack_edges)/G.number_of_edges():.2f}%)")
print(f"Distinct nodes touching at least one attack edge: {len(attack_nodes)}")
print(f"Is the #1 hub ({hub}) an attack node? {hub in attack_nodes}")

attack_degrees = sorted((G.degree(n) for n in attack_nodes), reverse=True)
print(f"\nAttack-node degree - top 10: {attack_degrees[:10]}")
print(f"Attack-node degree - median: {attack_degrees[len(attack_degrees)//2]}")
print(f"Attack nodes with degree == 1: {sum(1 for d in attack_degrees if d==1)} / {len(attack_degrees)}")

# do any of the small, non-giant components contain attack activity?
non_giant_with_attacks = sum(
    1 for c in components[1:]
    if any(float(d.get("ActivityLabel", 0)) != 0 for _, _, d in G.subgraph(c).edges(data=True))
)
print(f"\nNon-giant components containing an attack edge: {non_giant_with_attacks} / {len(components)-1}")

**Precise findings:**
- Only 2.63% of all edges are attack-labeled, and none of the top-5
  busiest nodes (including the main hub) touch a single one - the busiest
  node in this dataset is not implicated in any recorded attack.
- Attack-node degree is itself skewed: most attack-touching nodes only
  appear once (degree 1, same as the graph overall), but a handful reach
  degree in the hundreds - those are worth a closer look, since a node
  with 390 attack-flagged connections is a very different case from one
  with a single flagged connection.
- Every attack edge lives inside the giant component - none of the small,
  disconnected fragments contain any labeled attack activity.

**Takeaway for the team:** raw degree centrality alone would not have
flagged these attack nodes - they are structurally unremarkable (mostly
degree 1, same as 96% of the graph). This is a concrete, defensible reason
*why* the project looks beyond degree to betweenness, PageRank, and
clustering coefficient - and eventually to the ML/GNN models in Section 3
of the project.

In [ ]:
top_attack_nodes = sorted(attack_nodes, key=lambda n: G.degree(n), reverse=True)[:5]
print("Highest-degree attack-touching nodes:")
for n in top_attack_nodes:
    print(f"  {n}: degree={G.degree(n)}")

G_attack_view = get_readable_subgraph(G.subgraph(attack_nodes | {n for a in top_attack_nodes for n in G.neighbors(a)}), max_nodes=40)
plot_graph(G_attack_view, title="Neighborhood around the highest-degree attack-touching nodes", max_nodes=40,
           save_path="figures/fig_07_attack_neighborhood.png")

## 8. Graph feature extraction

Computing exact betweenness centrality on all 41,073 nodes is expensive
(the exact algorithm is O(V·E)). We compute it - along with the other
four features - on the **300 highest-degree nodes**, which includes every
node that matters structurally (all hubs, all high-degree attack nodes)
while running in under a second. This is a deliberate, stated scope
choice, not a silent shortcut.

In [ ]:
G_features = get_readable_subgraph(G, max_nodes=300)
features = compute_all_features(G_features)
print(f"Computed features for {len(features)} nodes")
features.sort_values("betweenness_centrality", ascending=False).round(5).head(10)

In [ ]:
verify_features(G_features, features)

Both checks should read `True` - the handshake lemma (sum of degrees =
2x edges) and PageRank scores summing to ~1. If either is `False`, do not
trust the numbers above it.

In [ ]:
plot_feature_distribution(features, "pagerank", title="PageRank distribution (top-300 subgraph)",
                           save_path="figures/fig_08_pagerank_distribution.png")

## 9. Motif counts: triangles and clustering coefficient

A **motif** here means a small, recurring connection pattern. The simplest
non-trivial one is a **triangle** - three nodes all connected to each
other. `nx.triangles()` counts, for every node, how many triangles it
participates in; it uses an efficient algorithm and runs in well under a
second even on the full 41,073-node graph.

`nx.clustering()` (the fraction of a node's neighbors that are also
connected to each other) is normally computed directly, but on this
specific graph it is pathologically slow - past 44 seconds with no
result, because a handful of very high-degree hub nodes make the
neighbor-pair-checking blow up. Since clustering coefficient is
mathematically defined in terms of triangle counts, we derive it directly
from the (fast) triangle counts instead:

$$C(v) = \frac{2 \cdot \text{triangles}(v)}{\deg(v)\,(\deg(v)-1)} \quad \text{for } \deg(v) \ge 2, \text{ else } 0$$

This produces results **identical** to `nx.clustering()` (verified below
on a subgraph small enough to run both), just without the slow path.

In [ ]:
triangles = nx.triangles(G)
total_triangles = sum(triangles.values()) // 3
nodes_in_triangles = sum(1 for v in triangles.values() if v > 0)

print(f"Total triangles in the full graph: {total_triangles}")
print(f"Nodes participating in at least one triangle: {nodes_in_triangles} / {G.number_of_nodes()}")

top5_triangle_nodes = sorted(triangles.items(), key=lambda x: x[1], reverse=True)[:5]
print("\nTop 5 nodes by triangle count:")
for node, count in top5_triangle_nodes:
    print(f"  {node}: {count} triangles, degree={G.degree(node)}, is_malicious={G.nodes[node].get('is_malicious')}")

**Only 47 triangles exist in the entire graph**, involving 52 of the
41,073 nodes (0.13%). This is exactly what the hub-and-leaf structure
from Section 3 predicts: a triangle needs three *mutually* connected
nodes, but 95.8% of nodes have degree 1 and can't be part of any
triangle at all. The triangles that do exist sit on a small set of
moderate-degree nodes near Section 5's "second tier" structure, not on
the dominant hub itself. Notably, `147.32.80.9` - the 4th busiest
triangle-forming node - **is** attack-labeled, unlike the top overall
hub.

In [ ]:
plot_bar(
    [n for n, _ in top5_triangle_nodes],
    [c for _, c in top5_triangle_nodes],
    title="Top 5 nodes by triangle count",
    xlabel="Triangles",
    legend_label="Node IP address (bar length = triangle count)",
    save_path="figures/fig_09_top_triangle_nodes.png",
)

In [ ]:
degrees_full = dict(G.degree())
clustering_coefficient = {
    v: (2 * triangles[v]) / (degrees_full[v] * (degrees_full[v] - 1)) if degrees_full[v] >= 2 else 0.0
    for v in G.nodes()
}

# Verify the derived formula itself is correct (not just fast) by comparing
# it against nx.clustering() computed on the SAME small subgraph, using
# that subgraph's own triangle counts and degrees - not the full graph's.
# (The 300-node feature subgraph is small enough for the slow direct
# algorithm to finish, unlike the full 41,073-node graph.) Comparing full-
# graph clustering values against a subgraph's nx.clustering() would not
# be a fair test: a node's degree and triangle count both change once
# edges outside the subgraph are dropped, so the two numbers describe
# different graphs, not the same one.
triangles_subgraph = nx.triangles(G_features)
degrees_subgraph = dict(G_features.degree())
derived_subgraph_clustering = {
    v: (2 * triangles_subgraph[v]) / (degrees_subgraph[v] * (degrees_subgraph[v] - 1)) if degrees_subgraph[v] >= 2 else 0.0
    for v in G_features.nodes()
}
nx_clustering_check = nx.clustering(G_features)
max_diff = max(abs(derived_subgraph_clustering[v] - nx_clustering_check[v]) for v in G_features.nodes())
print(f"Max difference between derived formula and nx.clustering(), both computed on the 300-node check subgraph: {max_diff}")

nonzero = [c for c in clustering_coefficient.values() if c > 0]
print(f"\nNodes with nonzero clustering coefficient: {len(nonzero)} / {G.number_of_nodes()}")
print(f"Mean clustering coefficient among those {len(nonzero)} nodes: {sum(nonzero)/len(nonzero):.4f}")
print(f"Mean clustering coefficient over the whole graph (including zeros): {sum(clustering_coefficient.values())/G.number_of_nodes():.6f}")

**The max difference above should read `0.0`** - the derived formula and
`nx.clustering()` agree exactly, they're the same quantity computed two
different ways. The whole-graph average is close to zero (0.0002) purely
because 99.87% of nodes have no triangle at all; among the 52 nodes that
*do* sit in a triangle, the average local clustering is a much more
typical 0.13.

## 10. Node2Vec embeddings

[Node2Vec](https://snap.stanford.edu/node2vec/) learns a dense vector for
every node from biased random walks, so that structurally similar nodes
end up with similar vectors - a learned alternative to hand-designed
features like degree or PageRank.

**Running Node2Vec on the full 41,073-node graph times out** (past 44
seconds with no result). The cause is the same dominant hub from Section
3: node2vec's 2nd-order random walks need to precompute a transition
probability for every pair of edges around each node, and a node with
degree 27,177 makes that precomputation combinatorially expensive.

A random sample of nodes doesn't fix this either - it makes it worse.
95.8% of nodes have degree 1, so a uniform random sample of, say, 1,000
nodes would mostly contain nodes with no edges *to each other* at all,
producing a nearly edgeless subgraph that random walks can't meaningfully
traverse.

Instead we scope Node2Vec to a subgraph that is guaranteed to have real,
connected topology: every attack-touching node from Section 7, plus all
of their immediate neighbors. This keeps every attack node's actual
local structure intact (which a random sample would destroy) while
excluding the problematic mega-hub.

In [ ]:
attack_neighborhood_nodes = set(attack_nodes)
for n in attack_nodes:
    attack_neighborhood_nodes.update(G.neighbors(n))

G_ml = G.subgraph(attack_neighborhood_nodes).copy()
ml_malicious_count = sum(1 for n in G_ml.nodes() if G_ml.nodes[n].get("is_malicious"))

print(f"Node2Vec / model subgraph: {G_ml.number_of_nodes()} nodes, {G_ml.number_of_edges()} edges")
print(f"Malicious nodes in this subgraph: {ml_malicious_count} / {G_ml.number_of_nodes()} ({100*ml_malicious_count/G_ml.number_of_nodes():.1f}%)")
print(f"Max degree in this subgraph: {max(dict(G_ml.degree()).values())} (no mega-hub - the full graph's top hub isn't an attack node, see Section 7)")
print("\nNOTE: this subgraph is deliberately built around attack activity, so its malicious rate "
      f"({100*ml_malicious_count/G_ml.number_of_nodes():.1f}%) is far higher than the full graph's true rate "
      f"({100*len(attack_nodes)/G.number_of_nodes():.2f}%). It is scoped this way on purpose, for the reasons "
      "above - it is not a representative sample of overall network traffic, and Section 11's results below "
      "should be read with that in mind, not as an estimate of real-world detection rates.")

In [ ]:
from node2vec import Node2Vec

node2vec_model = Node2Vec(
    # workers=1 to minimize (though, per gensim, not fully eliminate) run-to-run
    # randomness - see the note below the results table in Section 11.
    G_ml, dimensions=16, walk_length=10, num_walks=10, workers=1, quiet=True, seed=42
)
n2v_fit = node2vec_model.fit(window=5, min_count=1, seed=42)

ml_node_list = list(G_ml.nodes())
embeddings = pd.DataFrame(
    [n2v_fit.wv[n] for n in ml_node_list],
    index=ml_node_list,
    columns=[f"n2v_{i}" for i in range(16)],
)
print(f"Embedding matrix: {embeddings.shape}")
embeddings.head(3)

In [ ]:
from sklearn.decomposition import PCA

pca = PCA(n_components=2, random_state=42)
coords = pca.fit_transform(embeddings.values)
is_mal = [G_ml.nodes[n].get("is_malicious") for n in ml_node_list]

plot_scatter_2d(
    coords[:, 0], coords[:, 1], is_mal,
    title="Node2Vec embeddings (PCA to 2D) - attack neighborhood subgraph",
    xlabel=f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% variance)",
    ylabel=f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% variance)",
    save_path="figures/fig_10_node2vec_pca.png",
)

Two principal components only capture a fraction of a 16-dimensional
embedding's structure, so treat this plot as a rough sanity check, not
proof of separability - the classifiers in Section 11 use all 16
dimensions and substantially outperform what's visible here.

## 11. Models: Random Forest and XGBoost

We train two classifiers to predict `is_malicious` for each node in the
`G_ml` subgraph from Section 10, using:

- the five features from Section 8 (`compute_all_features`, now computed
  on `G_ml` instead of the top-300 subgraph): degree, closeness and
  betweenness centrality, PageRank, and clustering coefficient
- triangle count (Section 9)
- the 16-dimensional Node2Vec embedding (Section 10)

**On class balance:** `G_ml` is roughly 70% malicious by construction
(Section 10), far above the true ~1.6% base rate in the full graph. We
did not rebalance the classes - the goal here is to demonstrate that
graph features carry real predictive signal for these nodes, not to
produce a deployment-ready detector. A **majority-class baseline** is
included below specifically so the Random Forest / XGBoost numbers can
be read relative to a "did nothing" reference point, not as
free-standing accuracy figures.

GCN, GraphSAGE, and GAT are trained separately in Section 12, on the same
subgraph and split, so their results sit in the same comparison table.

In [ ]:
G_ml_features = compute_all_features(G_ml)
G_ml_features["triangle_count"] = pd.Series(nx.triangles(G_ml))
print(f"Computed features for {len(G_ml_features)} nodes")
print(verify_features(G_ml, G_ml_features))

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from xgboost import XGBClassifier

X = G_ml_features.drop(columns=["is_malicious"]).join(embeddings)
y = G_ml_features["is_malicious"].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, stratify=y, random_state=42
)
print(f"Train: {len(X_train)} nodes ({y_train.mean()*100:.1f}% malicious)")
print(f"Test:  {len(X_test)} nodes ({y_test.mean()*100:.1f}% malicious)")

models = {
    "Majority-class baseline": DummyClassifier(strategy="most_frequent", random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=200, random_state=42),
    "XGBoost": XGBClassifier(n_estimators=200, random_state=42, eval_metric="logloss"),
}

rows = []
fitted = {}
for name, clf in models.items():
    clf.fit(X_train, y_train)
    pred = clf.predict(X_test)
    proba = clf.predict_proba(X_test)[:, 1] if hasattr(clf, "predict_proba") else pred
    rows.append({
        "model": name,
        "accuracy": accuracy_score(y_test, pred),
        "f1": f1_score(y_test, pred),
        "roc_auc": roc_auc_score(y_test, proba),
    })
    fitted[name] = clf

results = pd.DataFrame(rows).set_index("model").round(4)
results

In [ ]:
plot_model_comparison(results, title="Random Forest vs. XGBoost vs. majority-class baseline",
                       save_path="figures/fig_11_model_comparison.png")

**Reading this table:** the majority-class baseline gets ~70% accuracy
for free, simply by always predicting "malicious" - that's the number to
beat, not 50%. Its ROC-AUC of 0.5 (chance level) is the more honest
baseline metric, since accuracy alone is misleading under class
imbalance. Both Random Forest and XGBoost clearly beat the baseline on
every metric, including ROC-AUC, which means the improvement reflects
real predictive signal in the graph features - not just the imbalance
itself.

**On exact reproducibility:** every other number in this notebook is
exactly reproducible (fixed seeds throughout). Node2Vec is a partial
exception - gensim's underlying Word2Vec training has a source of
run-to-run randomness that a fixed seed alone does not fully eliminate,
so re-running this notebook may shift the accuracy/F1/ROC-AUC figures
above by a few percentage points. Across repeated runs during
development, Random Forest and XGBoost consistently scored in the
0.92-0.96 range on accuracy and 0.93-0.96 on ROC-AUC, always far above
the 0.50 baseline - the conclusion (graph features carry real signal) is
stable even though the exact decimal is not.

In [ ]:
importances = pd.Series(
    fitted["Random Forest"].feature_importances_, index=X.columns
).sort_values(ascending=False).head(10)

plot_bar(
    importances.index[::-1], importances.values[::-1],
    title="Top 10 most important features (Random Forest)",
    xlabel="Feature importance",
    legend_label="Random Forest feature importance (Gini-based)",
    save_path="figures/fig_11_feature_importance.png",
)

Node2Vec dimensions dominate the top of the importance ranking, ahead of
every hand-designed centrality feature. That's a meaningful result on
its own: the learned embedding is picking up structural signal - about
*where in the graph* a node sits relative to attack activity - that
raw degree, PageRank, and clustering coefficient don't fully capture.

## 12. Models: Graph Neural Networks (GCN, GraphSAGE, GAT)

Random Forest and XGBoost (Section 11) cannot see graph structure directly -
that is exactly why Node2Vec had to hand them a pre-computed embedding as an
explicit feature. **Graph neural networks (GNNs) remove that middle step**:
each layer lets a node collect information directly from its neighbors
("message passing"), so structural learning happens end-to-end, inside the
classifier itself. To make that contrast fair and visible, the three GNNs
below are trained on the five centrality + motif features from Sections 8-9
only - **no Node2Vec embedding is fed in here**, since handing them one
would defeat the point of comparing "structure via a hand-designed
embedding" against "structure learned directly."

Three flavors, in plain English:
- **GCN** (Graph Convolutional Network) - a node's new representation is a
  weighted average of its neighbors' representations. Simple, and often a
  strong baseline.
- **GraphSAGE** - the same idea, but neighbors are combined through a
  separate learned transformation rather than a fixed averaging rule, and
  the architecture is designed to generalize to nodes it never saw during
  training.
- **GAT** (Graph Attention Network) - like GCN, but learns *how much
  attention* to pay to each neighbor instead of weighting every neighbor
  equally, since not every neighbor is equally informative.

All three use the **exact same** `G_ml` subgraph, train/test split, and
non-embedding features as Section 11, so the accuracy/F1/ROC-AUC numbers
below sit in the same table as Random Forest and XGBoost, directly
comparable.

**A caveat specific to this section, in the same spirit of honesty as the
rest of this notebook:** GCN/GraphSAGE/GAT need `torch` and
`torch-geometric`. The sandboxed environment used to build this notebook
could not fully install and execution-test them - its network policy blocks
the official CPU-only PyTorch package index, and falls back to resolving a
several-gigabyte CUDA-bundled build that does not fit the time/disk
available there. On a normal machine (yours, or the professor's),
`pip install torch` (or the explicit CPU-only command in
`requirements.txt`) pulls a small, fast CPU wheel with none of that problem.
The code below follows standard, well-established PyTorch Geometric
patterns and was checked carefully line by line; the parts that do not
need `torch` (building the graph's tensors) were separately verified with
plain NumPy. Please run this section once on your own machine and flag
anything that errors - exactly the same "run it, tell me what breaks"
process used for every other bug fixed in this project.

In [ ]:
try:
    import torch
    import torch.nn.functional as F
    from torch_geometric.data import Data
    from torch_geometric.nn import GCNConv, SAGEConv, GATConv
except ImportError as e:
    raise ImportError(
        "torch / torch-geometric not found. Install the CPU build with:\n"
        "  pip install torch --index-url https://download.pytorch.org/whl/cpu\n"
        "  pip install torch-geometric\n"
        f"(original error: {e})"
    )

import numpy as np
from sklearn.preprocessing import StandardScaler

gnn_nodes = list(G_ml.nodes())
gnn_node_idx = {n: i for i, n in enumerate(gnn_nodes)}
num_gnn_nodes = len(gnn_nodes)

# Every edge has to appear in both directions in edge_index - GCN/SAGE/GAT
# all aggregate FROM a node's neighbors, so a neighbor relationship needs to
# be visible from both ends, not just the direction it happened to be
# stored in the (undirected) networkx graph.
gnn_edges = list(G_ml.edges())
src = [gnn_node_idx[u] for u, v in gnn_edges] + [gnn_node_idx[v] for u, v in gnn_edges]
dst = [gnn_node_idx[v] for u, v in gnn_edges] + [gnn_node_idx[u] for u, v in gnn_edges]
edge_index_np = np.array([src, dst], dtype=np.int64)

# Same five centrality features + triangle count as Section 11, minus the
# Node2Vec embedding (see markdown above for why).
gnn_feature_cols = [c for c in G_ml_features.columns if c != "is_malicious"]
X_gnn = StandardScaler().fit_transform(G_ml_features.loc[gnn_nodes, gnn_feature_cols].values)
y_gnn = G_ml_features.loc[gnn_nodes, "is_malicious"].astype(int).values

# Reuse the IDENTICAL train/test split from Section 11 (same node IDs), so
# every model in this notebook is compared on the same held-out nodes -
# not just "a" 70/30 split, but the same one.
train_node_set = set(X_train.index)
test_node_set = set(X_test.index)
train_mask = np.array([n in train_node_set for n in gnn_nodes])
test_mask = np.array([n in test_node_set for n in gnn_nodes])
assert train_mask.sum() + test_mask.sum() == num_gnn_nodes

print(f"edge_index shape: {edge_index_np.shape} (2 rows x 2*edges, since edges are stored in both directions)")
print(f"feature matrix shape: {X_gnn.shape}")
print(f"train nodes: {train_mask.sum()}, test nodes: {test_mask.sum()} (identical split to Section 11)")

In [ ]:
torch_device = torch.device("cpu")  # these graphs are small enough that CPU is fine - no GPU needed

data = Data(
    x=torch.tensor(X_gnn, dtype=torch.float),
    edge_index=torch.tensor(edge_index_np, dtype=torch.long),
    y=torch.tensor(y_gnn, dtype=torch.long),
)
data.train_mask = torch.tensor(train_mask, dtype=torch.bool)
data.test_mask = torch.tensor(test_mask, dtype=torch.bool)


class GCN(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels):
        super().__init__()
        self.conv1 = GCNConv(in_channels, hidden_channels)
        self.conv2 = GCNConv(hidden_channels, out_channels)

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index).relu()
        x = F.dropout(x, p=0.5, training=self.training)
        return self.conv2(x, edge_index)


class GraphSAGE(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels):
        super().__init__()
        self.conv1 = SAGEConv(in_channels, hidden_channels)
        self.conv2 = SAGEConv(hidden_channels, out_channels)

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index).relu()
        x = F.dropout(x, p=0.5, training=self.training)
        return self.conv2(x, edge_index)


class GAT(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels, heads=4):
        super().__init__()
        self.conv1 = GATConv(in_channels, hidden_channels, heads=heads, dropout=0.6)
        self.conv2 = GATConv(hidden_channels * heads, out_channels, heads=1, concat=False, dropout=0.6)

    def forward(self, x, edge_index):
        x = F.dropout(x, p=0.6, training=self.training)
        x = self.conv1(x, edge_index).elu()
        x = F.dropout(x, p=0.6, training=self.training)
        return self.conv2(x, edge_index)

In [ ]:
def train_gnn(model, data, epochs=200, lr=0.01, weight_decay=5e-4, seed=42):
    torch.manual_seed(seed)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    model.train()
    for _ in range(epochs):
        optimizer.zero_grad()
        out = model(data.x, data.edge_index)
        loss = F.cross_entropy(out[data.train_mask], data.y[data.train_mask])
        loss.backward()
        optimizer.step()
    return model, loss.item()


def evaluate_gnn(model, data):
    model.eval()
    with torch.no_grad():
        out = model(data.x, data.edge_index)
        proba = F.softmax(out, dim=1)[:, 1].numpy()
        pred = out.argmax(dim=1).numpy()
    test_np = data.test_mask.numpy()
    y_true = data.y.numpy()[test_np]
    return {
        "accuracy": accuracy_score(y_true, pred[test_np]),
        "f1": f1_score(y_true, pred[test_np]),
        "roc_auc": roc_auc_score(y_true, proba[test_np]),
    }


in_channels = data.num_node_features
hidden_channels = 16
out_channels = 2  # two logits (benign, malicious); metrics use argmax and softmax[:, 1]

gnn_results, gnn_final_loss = {}, {}
for name, ModelClass in [("GCN", GCN), ("GraphSAGE", GraphSAGE), ("GAT", GAT)]:
    torch.manual_seed(42)
    model = ModelClass(in_channels, hidden_channels, out_channels)
    model, final_loss = train_gnn(model, data)
    gnn_results[name] = evaluate_gnn(model, data)
    gnn_final_loss[name] = final_loss

gnn_results_df = pd.DataFrame(gnn_results).T[["accuracy", "f1", "roc_auc"]].round(4)
print("Final training loss per model:", {k: round(v, 4) for k, v in gnn_final_loss.items()})
gnn_results_df

**Reading this table.** All three GNNs are trained for a fixed 200 epochs
with the Adam optimizer (`lr=0.01`), evaluated on the identical held-out
test nodes as every other model in this notebook. As with Node2Vec
(Section 10), GNN training has its own run-to-run randomness beyond what a
fixed seed fully controls, so exact numbers may shift slightly on re-run -
what matters is whether all three clearly beat the majority-class baseline
from Section 11 (69.6% accuracy, 0.50 ROC-AUC by definition), the same bar
every model in this notebook is measured against.

In [ ]:
all_model_results = pd.concat([results, gnn_results_df])
plot_model_comparison(
    all_model_results,
    title="All five models: accuracy / F1 / ROC-AUC",
    save_path="figures/fig_12_all_models_comparison.png",
)

## Summary - what to say in the presentation

- This dataset is three views of the **same** 41,073-node, ~45K-edge
  graph: "plain" keeps one edge per IP pair (last flow wins), "aggregated"
  also keeps one edge per pair but combines repeated flows using an
  unspecified (not simple-average) rule, and "multi" keeps all 100,000
  original flows as separate parallel edges.
- The graph is **hub-dominated at every scale**: one node touches 30% of
  all edges, 95.8% of nodes have degree exactly 1, and even small,
  fully-disconnected fragments repeat the same one-to-many star shape.
  This is consistent with a single-vantage-point NetFlow capture.
- Attacks are **not** concentrated on the busiest nodes - the top 5 hubs,
  including the dominant one, touch zero attack-labeled edges. Attack
  nodes are mostly structurally unremarkable (degree 1), which is the
  concrete justification for using centrality features beyond raw degree,
  and eventually ML/GNN models, rather than a simple degree threshold.
- **Motifs are rare**: only 47 triangles exist in the whole graph,
  involving 0.13% of nodes - a direct, quantified consequence of the
  hub-and-leaf structure above. Clustering coefficient was derived from
  triangle counts (verified identical to `nx.clustering()`) rather than
  computed directly, since the direct algorithm is pathologically slow
  on this specific hub-dominated graph.
- **Node2Vec** does not run on the full graph in reasonable time (same
  mega-hub problem), and a random node sample would be too sparse to
  form a connected subgraph (95.8% degree-1 nodes). It was instead
  scoped to the 962-node neighborhood around all attack-touching nodes -
  a deliberate, stated choice, not a silent shortcut.
- **Random Forest and XGBoost**, trained on that same 962-node
  neighborhood using centrality + motif + Node2Vec features, clearly
  beat a majority-class baseline on accuracy, F1, and ROC-AUC. Node2Vec
  dimensions were the single most important feature group, ahead of
  every hand-designed centrality feature - the learned embedding
  captures structural signal the others miss. This subgraph is ~70%
  malicious by construction (not the true ~1.6% base rate), so these
  numbers demonstrate that graph features carry real signal, not that
  this is a deployment-ready detector.
- All numbers in this notebook come from the complete 41,073-node graph
  (no sampling) except: the centrality features in Section 8 (scoped to
  the top-300 nodes), and Node2Vec / Section 11's models (scoped to the
  962-node attack neighborhood) - both for stated, compute-time reasons.
- **GCN, GraphSAGE, and GAT** (Section 12) were added to train directly on
  graph structure via message passing, using the same features as Random
  Forest/XGBoost minus Node2Vec (so the comparison isolates "hand-designed
  embedding" vs. "learned end-to-end"), on the identical subgraph and
  train/test split as every other model. Unlike everything else in this
  notebook, this section could not be execution-tested in the sandboxed
  environment used to build it (see Section 12 for the specific,
  infrastructure-only reason) - run it once before presenting and treat any
  error the same as any other bug in this project.
